# M365 Work Behavior — Direct Ingester (Fabric)

Pulls **manual** work activity from the Microsoft 365 Unified Audit Log (Exchange, SharePoint, OneDrive, Teams, OneNote, Planner, Forms, Power BI/Automate), classifies each record into a **Behavior_Category** aligned to the dashboard's Signal Map, and writes an aggregated Delta table `dbo.m365_work_behavior`.

This is the manual-work counterpart to the Copilot Audit ingester and powers the **AI vs Manual Work** page. Same Graph auth / staging / resume infrastructure.

**Prereqs:** an Entra app registration with `AuditLog.Read.All` (application) + admin consent; Unified Audit Log enabled in the tenant.

## 1. Configuration

In [ ]:
# === CONFIG — M365 Work Behavior Ingester ===
# Pulls MANUAL work activity from the M365 Unified Audit Log (Exchange, SharePoint,
# OneDrive, Teams, OneNote, Planner, Forms, Power BI/Automate) via the Microsoft Graph
# security/auditLog API, maps each record to a Behavior_Category (aligned to the
# dashboard's Signal Map), and writes an aggregated Delta table the PBIT consumes.
#
# This is the "manual-work counterpart" to the Copilot Audit ingester: same auth,
# staging and resume infrastructure, different record types + a behavior classifier.

TENANT_ID     = '<your-tenant-guid>'                  # Entra -> Overview -> Tenant ID
CLIENT_ID     = '<your-app-reg-client-id>'            # Entra -> App registrations -> your app -> Application (client) ID
CLIENT_SECRET = '<your-client-secret-value>'          # The secret VALUE. Prefer Key Vault (see below).

# --- Load mode ---------------------------------------------------------------------
# 'backfill'    = pull BACKFILL_DAYS of history once (overwrite). Use for the initial load.
# 'incremental' = pull only since the latest EventDate already in OUTPUT_TABLE (append). Daily/weekly.
MODE          = 'incremental'
BACKFILL_DAYS = 30                                    # UAL retention is ~90d (E3) / 180d (E5); manual volume is high, keep modest.
LOOKBACK_DAYS = 7
OUTPUT_TABLE  = 'dbo.m365_work_behavior'              # Delta table consumed by the M365 Signals PBIT (keep name/schema stable)

# --- Manual-work record types (Graph audit beta recordTypeFilters, camelCase) -------
# These are the M365 workloads that have a genuine manual equivalent to Copilot actions.
RECORD_TYPES  = [
    'exchangeItem', 'exchangeItemAggregated', 'exchangeItemGroup',
    'sharePointFileOperation', 'sharePoint', 'sharePointSharingOperation',
    'oneDrive', 'microsoftTeams', 'oneNote',
    'microsoftForms', 'microsoftFlow', 'powerBIAudit', 'planner',
]

# --- Backfill scale tuning ---------------------------------------------------------
CHUNK_HOURS             = 6      # manual UAL is high-volume; smaller windows = finer parallelism + resume.
MAX_CONCURRENT_QUERIES  = 6
MAX_WAIT_MIN_PER_QUERY  = 240
POLL_INTERVAL_SEC       = 30
STAGING_DIR             = 'Files/_m365_work_staging'   # per-window JSONL + manifest; enables resume.

# --- Production: load secret from Key Vault instead of hardcoding -------------------
# from notebookutils.credentials import getSecret
# CLIENT_SECRET = getSecret('https://<your-vault>.vault.azure.net/', 'M365WorkAppSecret')


## 2. Authenticate to Microsoft Graph

In [ ]:
import requests, time
from requests.adapters import HTTPAdapter
try:
    from urllib3.util.retry import Retry
except Exception:
    from requests.packages.urllib3.util.retry import Retry

# Resilient HTTP session. The Microsoft Graph security/auditLog endpoints intermittently return
# 502/503/504 (gateway timeouts) and 429 (throttling), especially during the poll + records-fetch
# phases on busy tenants. A shared Session with urllib3 Retry transparently retries those with
# exponential backoff + Retry-After, so a transient blip no longer aborts the whole run.
def _build_session() -> requests.Session:
    s = requests.Session()
    retry = Retry(
        total=8,                       # up to 8 attempts per request
        connect=5, read=5, status=8,
        backoff_factor=2,              # 0,2,4,8,16,32... seconds (capped by urllib3)
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(['GET', 'POST']),
        respect_retry_after_header=True,
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=8, pool_maxsize=8)
    s.mount('https://', adapter)
    s.mount('http://', adapter)
    return s

SESSION = _build_session()

# Belt-and-braces wrapper: urllib3 Retry handles most cases, but we add an outer retry loop so a
# request that *still* surfaces a transient status (or a connection error) after the adapter gives
# up is retried a few more times rather than killing the notebook.
_TRANSIENT = {429, 500, 502, 503, 504}

def _request(method: str, url: str, *, attempts: int = 5, **kwargs):
    kwargs.setdefault('timeout', 60)
    last = None
    for i in range(1, attempts + 1):
        try:
            r = SESSION.request(method, url, **kwargs)
        except requests.exceptions.RequestException as e:
            last = e
            if i == attempts:
                raise
            wait = min(60, 2 ** i)
            print(f'    transient network error ({type(e).__name__}); retry {i}/{attempts} in {wait}s')
            time.sleep(wait)
            continue
        if r.status_code in _TRANSIENT and i < attempts:
            wait = min(60, 2 ** i)
            print(f'    transient HTTP {r.status_code} on {method} {url.split("?")[0].rsplit("/",1)[-1]}; '
                  f'retry {i}/{attempts} in {wait}s')
            time.sleep(wait)
            continue
        return r
    return r  # pragma: no cover

# Auto-refreshing app-only Graph token. Long (multi-hour) chunked runs would otherwise
# fail when the 1-hour token expires mid-pull, so we transparently refresh every ~50 min.
def _get_graph_token() -> str:
    url  = f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token'
    body = {
        'client_id':     CLIENT_ID,
        'scope':         'https://graph.microsoft.com/.default',
        'client_secret': CLIENT_SECRET,
        'grant_type':    'client_credentials',
    }
    r = _request('POST', url, data=body, timeout=30)
    r.raise_for_status()
    return r.json()['access_token']

_token = {'value': None, 'ts': 0.0}

def get_headers() -> dict:
    if _token['value'] is None or (time.time() - _token['ts']) > 50 * 60:
        _token['value'] = _get_graph_token()
        _token['ts'] = time.time()
    return {'Authorization': f'Bearer {_token["value"]}', 'Content-Type': 'application/json'}

get_headers()
print('\u2713 Graph token acquired (auto-refreshing, retry-resilient session).')


## 3. Build query windows + the `create_query` helper

In [ ]:
from datetime import datetime, timezone, timedelta

# Same windowing model as the Copilot audit ingester: incremental reads the high-water
# mark (max EventDate) from OUTPUT_TABLE so daily runs stay cheap; backfill pulls
# BACKFILL_DAYS at once and overwrites.
end_date = datetime.now(timezone.utc)
if MODE == 'backfill':
    start_date = end_date - timedelta(days=BACKFILL_DAYS)
    WRITE_MODE = 'overwrite'
else:
    WRITE_MODE = 'append'
    try:
        from pyspark.sql import functions as F
        _hw = spark.table(OUTPUT_TABLE).agg(F.max('EventDate').alias('m')).collect()[0]['m']
    except Exception:
        _hw = None
    if _hw:
        from datetime import datetime as _dt
        start_date = (_hw if isinstance(_hw, datetime) else _dt.fromisoformat(str(_hw))).replace(tzinfo=timezone.utc)
        print(f'incremental: high-water mark = {start_date:%Y-%m-%d} (table {OUTPUT_TABLE})')
    else:
        start_date = end_date - timedelta(days=LOOKBACK_DAYS)
        print(f'incremental: no prior data, defaulting to last {LOOKBACK_DAYS}d')

_cur = start_date.replace(hour=0, minute=0, second=0, microsecond=0)  # midnight anchor => stable window keys for resume
windows = []
while _cur < end_date:
    _nxt = min(_cur + timedelta(hours=CHUNK_HOURS), end_date)
    windows.append((_cur, _nxt))
    _cur = _nxt
print(f'MODE={MODE}: {start_date:%Y-%m-%d} -> {end_date:%Y-%m-%d}  =>  {len(windows)} window(s) of {CHUNK_HOURS}h, WRITE_MODE={WRITE_MODE}')

def create_query(win_start, win_end) -> str:
    body = {
        'displayName':         f'M365 Work {win_start:%Y%m%d%H%M}-{win_end:%Y%m%d%H%M}',
        'filterStartDateTime': win_start.isoformat(),
        'filterEndDateTime':   win_end.isoformat(),
        'recordTypeFilters':   RECORD_TYPES,
        'operationFilters':    [],
    }
    r = _request('POST', 'https://graph.microsoft.com/beta/security/auditLog/queries',
                 json=body, headers=get_headers(), timeout=30)
    r.raise_for_status()
    return r.json()['id']


## 4. Resumable wait helper

In [ ]:
import time

def wait_for_query(query_id: str, max_wait_min: int = MAX_WAIT_MIN_PER_QUERY) -> None:
    started  = time.time()
    deadline = started + max_wait_min * 60
    polls = 0
    while time.time() < deadline:
        polls += 1
        r = _request('GET',
            f'https://graph.microsoft.com/beta/security/auditLog/queries/{query_id}',
            headers=get_headers(), timeout=30)
        r.raise_for_status()
        status = r.json()['status']
        if status == 'succeeded':
            return
        if status in ('failed', 'cancelled'):
            raise RuntimeError(f'Query {query_id} ended with status: {status}')
        if polls % 4 == 0:
            mins = int((time.time() - started) // 60)
            print(f'    … still {status} ({mins}m elapsed, {polls} polls)')
        time.sleep(POLL_INTERVAL_SEC)
    raise TimeoutError(
        f'Query {query_id} did not succeed within {max_wait_min} min. '
        f'Increase MAX_WAIT_MIN_PER_QUERY or reduce CHUNK_HOURS.')


## 5. Fetch records → stream to Lakehouse Files (chunked, resumable)

In [ ]:
import os, json, time
from concurrent.futures import ThreadPoolExecutor, as_completed

# Resumable, bounded-concurrency backfill. Each window's records stream to a JSONL file and its
# status is recorded in a manifest; rerun skips windows already 'succeeded'. Wall-clock tracks the
# slowest concurrent window, not the serial sum - so 180 windows finish in a handful of waves
# instead of timing out one-by-one. Survives Fabric session timeouts (no clear unless backfill+empty).
STAGING_ABS  = '/lakehouse/default/' + STAGING_DIR.rstrip('/')
MANIFEST     = os.path.join(STAGING_ABS, '_manifest.json')
os.makedirs(STAGING_ABS, exist_ok=True)

def _key(ws, we): return f'{ws:%Y%m%d%H%M}_{we:%Y%m%d%H%M}'
def _load_manifest():
    try:
        with open(MANIFEST) as f: return json.load(f)
    except Exception:
        return {}
def _save_manifest(m):
    os.makedirs(STAGING_ABS, exist_ok=True)
    with open(MANIFEST, 'w') as f: json.dump(m, f)

manifest = _load_manifest()
# Fresh start only for an overwrite backfill with no prior progress; otherwise resume.
if MODE == 'backfill' and not manifest:
    for _fn in os.listdir(STAGING_ABS):
        try: os.remove(os.path.join(STAGING_ABS, _fn))
        except OSError: pass
    manifest = {}

def _row(r):
    return {'RecordId': r.get('id'), 'CreationDate': r.get('createdDateTime'),
            'RecordType': r.get('auditLogRecordType'), 'Operation': r.get('operation'),
            'AuditData': json.dumps(r.get('auditData', {})) if r.get('auditData') else None,
            'AssociatedAdminUnits': json.dumps(r.get('associatedAdminUnits', [])),
            'AssociatedAdminUnitsNames': json.dumps(r.get('associatedAdminUnitsNames', []))}

def _drain(qid, key):
    n, page = 0, 0
    url = f'https://graph.microsoft.com/beta/security/auditLog/queries/{qid}/records?$top=999'
    while url:
        r = _request('GET', url, headers=get_headers(), timeout=60); r.raise_for_status()
        d = r.json(); vals = d.get('value', [])
        if vals:
            with open(os.path.join(STAGING_ABS, f'win_{key}_{page:04d}.jsonl'), 'w', encoding='utf-8') as f:
                for rec in vals: f.write(json.dumps(_row(rec)) + '\n')
            page += 1; n += len(vals)
        url = d.get('@odata.nextLink')
    return n

def _process(win):
    ws, we = win; key = _key(ws, we)
    if manifest.get(key, {}).get('status') == 'succeeded':
        return key, manifest[key].get('rows', 0), 'skip'
    qid = create_query(ws, we)
    wait_for_query(qid)                     # per-window ceiling; bubbles up only that window on timeout
    n = _drain(qid, key)
    manifest[key] = {'status': 'succeeded', 'rows': n}; _save_manifest(manifest)
    return key, n, 'done'

pending = [w for w in windows if manifest.get(_key(*w), {}).get('status') != 'succeeded']
print(f'{len(windows)-len(pending)} window(s) already done; {len(pending)} to fetch (<= {MAX_CONCURRENT_QUERIES} at a time).')
total = sum(v.get('rows', 0) for v in manifest.values()); done = 0
with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_QUERIES) as ex:
    futs = {ex.submit(_process, w): w for w in pending}
    for fut in as_completed(futs):
        key, n, how = fut.result(); total += n if how == 'done' else 0; done += 1
        print(f'  [{done}/{len(pending)}] {key} {how} (+{n:,})')
print(f'Streamed {total:,} records across {len([f for f in os.listdir(STAGING_ABS) if f.endswith(".jsonl")])} file(s). Manifest: {MANIFEST}')


## 6. Parse `AuditData` → manual-work fields

In [ ]:
import os, glob
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

# Read every per-window JSONL the resumable fetch wrote. Distributed read.
_files = glob.glob('/lakehouse/default/' + STAGING_DIR.rstrip('/') + '/win_*.jsonl')
_cols = ['RecordId', 'CreationDate', 'RecordType', 'Operation', 'AuditData']
if not _files:
    raw = spark.createDataFrame([], StructType([StructField(c, StringType()) for c in _cols]))
else:
    raw = spark.read.json(STAGING_DIR.rstrip('/') + '/win_*.jsonl')
raw = raw.persist()
print(f'Raw DataFrame rows: {raw.count():,} (from {len(_files)} window file(s))')

# Minimal AuditData schema: user, workload, operation, and the source file (for extension-based classification).
audit_schema = StructType([
    StructField('CreationTime', StringType()),
    StructField('UserId', StringType()),
    StructField('Workload', StringType()),
    StructField('Operation', StringType()),
    StructField('RecordType', StringType()),
    StructField('SourceFileName', StringType()),
    StructField('SourceFileExtension', StringType()),
    StructField('ObjectId', StringType()),
])

parsed = (raw
    .filter(F.col('AuditData').isNotNull() & (F.length('AuditData') > 10))
    .withColumn('j', F.from_json('AuditData', audit_schema))
    .filter(F.col('j').isNotNull()))

flat = parsed.select(
    F.col('j.CreationTime').cast('timestamp').alias('CreationDate'),
    F.to_date(F.col('j.CreationTime')).alias('EventDate'),
    F.lower(F.trim(F.col('j.UserId'))).alias('UserId'),
    F.coalesce(F.col('j.Workload'), F.col('RecordType')).alias('Workload'),
    F.coalesce(F.col('j.Operation'), F.col('Operation')).alias('Operation'),
    F.col('RecordType').alias('RecordTypeName'),
    F.lower(F.coalesce(F.col('j.SourceFileExtension'),
                       F.regexp_extract(F.coalesce(F.col('j.SourceFileName'), F.lit('')), r'\.([A-Za-z0-9]+)$', 1))).alias('Ext'),
).filter(F.col('UserId').isNotNull() & (F.col('UserId') != ''))


## 7. Classify each record into a Behavior_Category + aggregate

In [ ]:
from pyspark.sql import functions as F

# Behavior classifier — mirrors the dashboard Signal Map. Maps (RecordType, Operation, file
# extension) to a Behavior_Category so manual M365 activity lines up 1:1 with Copilot behaviours
# on the "AI vs Manual Work" page. Order matters: first match wins.
op   = F.lower(F.coalesce(F.col('Operation'), F.lit('')))
rt   = F.lower(F.coalesce(F.col('RecordTypeName'), F.lit('')))
ext  = F.col('Ext')
wl   = F.lower(F.coalesce(F.col('Workload'), F.lit('')))

doc_ext   = ext.isin('docx','doc','rtf','odt')
xls_ext   = ext.isin('xlsx','xls','csv')
ppt_ext   = ext.isin('pptx','ppt')
pdf_ext   = (ext == 'pdf')
img_ext   = ext.isin('png','jpg','jpeg','gif','bmp','heic')
vid_ext   = ext.isin('mp4','mov','avi','wmv')
code_ext  = ext.isin('py','js','ts','java','cs','cpp','sql','json','yaml','yml','sh','ps1')
is_modify = op.rlike('modif|upload|create|update|checkin|add')
is_access = op.rlike('access|download|view|read|preview')

behavior = (F.when(rt.rlike('exchangeitem') & op.rlike('send|update|create'), 'Email Drafting')
    .when(rt.rlike('exchangeitemaggregated') | op.rlike('mailitemsaccessed'), 'Email Summarising')
    .when(rt.rlike('calendar') | op.rlike('meeting'), 'Meeting Scheduling')
    .when(rt.rlike('teams') & op.rlike('message|chat|post'), 'Teams Messaging')
    .when(rt.rlike('teams'), 'Meeting Prep')
    .when(rt.rlike('sharepointfileoperation') & doc_ext & is_modify, 'Document Drafting')
    .when(rt.rlike('sharepointfileoperation') & doc_ext & is_access, 'Document Summarising')
    .when(rt.rlike('sharepointfileoperation') & xls_ext & is_modify, 'Excel Assistance')
    .when(rt.rlike('sharepointfileoperation') & xls_ext & is_access, 'Spreadsheet Review')
    .when(rt.rlike('sharepointfileoperation') & ppt_ext & is_modify, 'Presentation Creation')
    .when(rt.rlike('sharepointfileoperation') & ppt_ext & is_access, 'Presentation Summarising')
    .when(rt.rlike('sharepointfileoperation') & pdf_ext, 'PDF Analysis')
    .when(rt.rlike('sharepointfileoperation') & img_ext, 'Image / Media Analysis')
    .when(rt.rlike('sharepointfileoperation') & vid_ext, 'Video Summarising')
    .when(rt.rlike('sharepointfileoperation') & code_ext & is_modify, 'Code Writing')
    .when(rt.rlike('sharepointfileoperation') & code_ext & is_access, 'Code Analysis')
    .when(op.rlike('searchqueryperformed|search'), 'Enterprise Searching')
    .when(rt.rlike('sharepointfileoperation') & op.rlike('download|accessed'), 'File Retrieval')
    .when(rt.rlike('sharepoint') & op.rlike('pageviewed|view'), 'SharePoint Access')
    .when(rt.rlike('onenote'), 'Note Taking')
    .when(rt.rlike('planner'), 'Task Management')
    .when(rt.rlike('forms'), 'Form / Survey Work')
    .when(rt.rlike('flow'), 'Running a Workflow')
    .when(rt.rlike('powerbi'), 'Data Querying')
    .when(rt.rlike('sharepointfileoperation') & is_modify, 'Document Drafting')
    .when(rt.rlike('sharepointfileoperation') & is_access, 'File Retrieval')
    .otherwise(F.initcap(F.regexp_replace(rt, '([a-z])([A-Z])', r'$1 $2'))))

classified = flat.withColumn('Behavior_Category', behavior)

# Aggregate to the PBIT contract: (UserId, Behavior_Category, Workload, Operation, EventDate, EventCount)
agg = (classified
    .filter(F.col('EventDate').isNotNull())
    .groupBy('UserId', 'Behavior_Category', 'Workload', 'Operation', 'EventDate')
    .agg(F.count(F.lit(1)).alias('EventCount')))

print('Behavior mix (top 15):')
agg.groupBy('Behavior_Category').agg(F.sum('EventCount').alias('events')).orderBy(F.desc('events')).show(15, truncate=False)
print(f'Aggregated rows: {agg.count():,}')


## 8. Write to Lakehouse Delta table

In [ ]:
# Write to the Lakehouse Delta table the M365 Signals PBIT consumes.
# Schema contract: UserId, Behavior_Category, Workload, Operation, EventDate, EventCount.
(agg.select('UserId', 'Behavior_Category', 'Workload', 'Operation', 'EventDate', 'EventCount')
    .write.format('delta').mode(WRITE_MODE).option('mergeSchema', 'true')
    .saveAsTable(OUTPUT_TABLE))
print(f'Wrote {OUTPUT_TABLE} (mode={WRITE_MODE}).')


## 9. Verify

In [ ]:
from pyspark.sql import functions as F
_t = spark.table(OUTPUT_TABLE)
print(f'{OUTPUT_TABLE}: {_t.count():,} rows, {_t.select("UserId").distinct().count():,} users')
_t.groupBy('Behavior_Category').agg(F.sum('EventCount').alias('events')).orderBy(F.desc('events')).show(20, truncate=False)


---
**Toggle:** the PBIT's `Enable_M365WorkPatterns` parameter gates this table. When set to `Exclude` (or the table is absent) the M365 pages show empty gracefully.